In [8]:
from langchain_pymupdf4llm import PyMuPDF4LLMLoader
import pprint

In [15]:
!pip install -qU langchain-openai

In [14]:
from langchain_community.document_loaders.parsers import LLMImageBlobParser
from langchain_openai import ChatOpenAI

ModuleNotFoundError: No module named 'langchain_openai'

In [ ]:
# fie path 
file_path = r"C:\Ask My Paper\Data\Copy of AI Engineering-1-150.pdf"

In [10]:
# load the PDF file using PyMuPDF4LLMLoader
loader = PyMuPDF4LLMLoader(file_path=file_path,
                           mode= "page",
                           page_delimiter="\n------------------Page-Ends-----------------\n\n",)

docs = loader.load()


Warning - arguments ignored in legacy mode: {'page_delimiter'}.
Warning - arguments ignored in legacy mode: {'page_delimiter'}.
Warning - arguments ignored in legacy mode: {'page_delimiter'}.
Warning - arguments ignored in legacy mode: {'page_delimiter'}.
Warning - arguments ignored in legacy mode: {'page_delimiter'}.
Warning - arguments ignored in legacy mode: {'page_delimiter'}.
Warning - arguments ignored in legacy mode: {'page_delimiter'}.
Warning - arguments ignored in legacy mode: {'page_delimiter'}.
Warning - arguments ignored in legacy mode: {'page_delimiter'}.
Warning - arguments ignored in legacy mode: {'page_delimiter'}.
Warning - arguments ignored in legacy mode: {'page_delimiter'}.
Warning - arguments ignored in legacy mode: {'page_delimiter'}.
Warning - arguments ignored in legacy mode: {'page_delimiter'}.
Warning - arguments ignored in legacy mode: {'page_delimiter'}.
Warning - arguments ignored in legacy mode: {'page_delimiter'}.
Warning - arguments ignored in legacy mo

In [11]:
docs[1].page_content

'“This book of fers a comprehensive, well-structured guide to the essential\n\naspects of building generative AI systems. A must-read for any professional\nlooking to scale AI across the enterprise.”\n\nVittorio Cretella, former global CIO at P&G and Mars\n\n\n“Chip Huyen gets generative AI. She is a remarkable teacher and writer\n\nwhose work has been instrumental in helping teams bring AI into production.\nDrawing on her deep expertise, _AI Engineering_ is a comprehensive and\nholistic guide to building generative AI applications in production.”\nLuke Metz, cocreator of ChatGPT, former research manager at OpenAI\n\n##### **AI Engineering**\n\n\nFoundation models have enabled many new AI use cases while lowering the barriers to entry for\nbuilding AI products. This has transformed AI from an esoteric discipline into a powerful development\ntool that anyone can use—including those with no prior AI experience.\n\nIn this accessible guide, author Chip Huyen discusses AI engineering: the 

In [12]:
pprint.pp(docs[0].metadata)

{'producer': 'iLovePDF',
 'creator': '',
 'creationdate': '',
 'source': 'C:\\Ask My Paper\\Data\\Copy of AI Engineering-1-150.pdf',
 'file_path': 'C:\\Ask My Paper\\Data\\Copy of AI Engineering-1-150.pdf',
 'total_pages': 150,
 'format': 'PDF 1.6',
 'title': '',
 'author': '',
 'subject': '',
 'keywords': '',
 'moddate': '2026-06-18T17:36:27+00:00',
 'trapped': '',
 'modDate': 'D:20260618173627Z',
 'creationDate': '',
 'page': 0}


In [ ]:
# for future use code for image extraction using multimodel 

import os
import fitz
from PIL import Image
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration
import torch

# -----------------------------
# Load Vision Model
# -----------------------------
model_name = "Qwen/Qwen2.5-VL-3B-Instruct"

processor = AutoProcessor.from_pretrained(model_name)

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)


# -----------------------------
# Image Caption Function
# -----------------------------
def describe_image(image_path):

    image = Image.open(image_path)

    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "image": image
                },
                {
                    "type": "text",
                    "text": (
                        "Describe this image in detail. "
                        "Mention diagrams, charts, tables, "
                        "objects and important information."
                    )
                }
            ]
        }
    ]

    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = processor(
        text=[text],
        images=[image],
        return_tensors="pt"
    ).to(model.device)

    output = model.generate(
        **inputs,
        max_new_tokens=200
    )

    response = processor.batch_decode(
        output,
        skip_special_tokens=True
    )[0]

    return response


# -----------------------------
# PDF Loader
# -----------------------------
class PDFLoader:

    def __init__(self, pdf_path):

        self.pdf_path = pdf_path

        os.makedirs("images", exist_ok=True)

    def load(self):

        document = fitz.open(self.pdf_path)

        data = []

        for page_number, page in enumerate(document, start=1):

            # -------------------------
            # Extract Text
            # -------------------------
            page_text = page.get_text()

            # -------------------------
            # Extract Images
            # -------------------------
            image_descriptions = []

            images = page.get_images(full=True)

            for index, img in enumerate(images):

                xref = img[0]

                base_image = document.extract_image(xref)

                image_bytes = base_image["image"]

                image_ext = base_image["ext"]

                image_path = (
                    f"images/page_{page_number}_{index}.{image_ext}"
                )

                with open(image_path, "wb") as f:
                    f.write(image_bytes)

                # Generate caption
                caption = describe_image(image_path)

                image_descriptions.append({
                    "image_path": image_path,
                    "caption": caption
                })

            # -------------------------
            # Store Page Information
            # -------------------------
            data.append({

                "page": page_number,

                "text": page_text,

                "images": image_descriptions,

                "source": self.pdf_path

            })

        document.close()

        return data


# -----------------------------
# Example
# -----------------------------

loader = PDFLoader("sample.pdf")

documents = loader.load()

for page in documents:

    print("=" * 60)

    print("PAGE:", page["page"])

    print("\nTEXT:\n")

    print(page["text"][:300])

    print("\nIMAGE DESCRIPTIONS:\n")

    for img in page["images"]:

        print(img["image_path"])

        print(img["caption"])

        print()